In [1]:
from pyspark.sql import SparkSession
import os
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("WeatherIntelligence")
    .master("local[*]")
    .getOrCreate()
)

print("Spark Version:", spark.version)
print("Spark Application:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

ModuleNotFoundError: No module named 'pyspark'

In [ ]:

project_dir = os.path.dirname(os.getcwd())
data_dir = os.path.join(project_dir, "Data")
raw_dir = os.path.join(data_dir, "raw")

print("Raw Data Directory:", raw_dir)
print("\nFiles:")
print(os.listdir(raw_dir))

In [ ]:
#Dataset Inspection
datasets = {
    "stations": os.path.join(raw_dir, "stations.csv"),
    "observations": os.path.join(raw_dir, "weather_observations.csv"),
    "forecasts": os.path.join(raw_dir, "weather_forecasts.csv"),
    "alerts": os.path.join(raw_dir, "weather_alerts.csv")
}

for name, path in datasets.items():

    print(f"\n{'=' * 70}")
    print(f"DATASET: {name.upper()}")
    print(f"{'=' * 70}")

    # Read CSV using Spark for initial inspection
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(path)
    )

    print("\nSchema:")
    df.printSchema()

    print(f"\nTotal Rows: {df.count()}")
    print(f"Total Columns: {len(df.columns)}")

    print("\nColumn Names:")
    print(df.columns)

    print("\nFirst 5 Records:")
    df.show(5, truncate=False)

In [ ]:
#Schema Definition and Data Ingestion
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)

# 1. Define schemas

stations_schema = StructType([
    StructField("station_id", StringType(), True),
    StructField("station_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("elevation_m", IntegerType(), True)
])

observations_schema = StructType([
    StructField("observation_id", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("observed_at", TimestampType(), True),
    StructField("temperature_c", DoubleType(), True),
    StructField("humidity_pct", DoubleType(), True),
    StructField("rainfall_mm", DoubleType(), True),
    StructField("wind_kph", DoubleType(), True),
    StructField("condition", StringType(), True)
])

forecasts_schema = StructType([
    StructField("forecast_id", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("forecast_for", TimestampType(), True),
    StructField("forecast_temp_c", DoubleType(), True),
    StructField("forecast_rain_mm", DoubleType(), True),
    StructField("forecast_condition", StringType(), True)
])

alerts_schema = StructType([
    StructField("alert_id", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("alert_time", TimestampType(), True),
    StructField("alert_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("status", StringType(), True)
])


# 2. Preserve raw datasets (all columns as strings)

raw_dfs = {
    name: (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(path)
    )
    for name, path in datasets.items()
}


# 3. Load datasets using explicit schemas

schemas = {
    "stations": stations_schema,
    "observations": observations_schema,
    "forecasts": forecasts_schema,
    "alerts": alerts_schema
}

dataframes = {
    name: (
        spark.read
        .option("header", True)
        .schema(schemas[name])
        .csv(datasets[name])
    )
    for name in datasets
}


# 4. Assign individual DataFrame variables

stations_df = dataframes["stations"]
observations_df = dataframes["observations"]
forecasts_df = dataframes["forecasts"]
alerts_df = dataframes["alerts"]


# 5. Verify loaded schemas

for name, df in dataframes.items():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()} SCHEMA")
    print(f"{'=' * 50}")
    df.printSchema()

## **Initial Data Quality Assessment**

In [ ]:
#Finding Missing Values in each raw dataset
for name, df in raw_dfs.items():

    print(f"\n{'=' * 60}")
    print(f"MISSING VALUE PROFILE: {name.upper()}")
    print(f"{'=' * 60}")

    total_rows = df.count()

    for column in df.columns:

        missing_count = df.filter(
            F.col(column).isNull()
            | (F.trim(F.col(column)) == "")
        ).count()

        if missing_count > 0:
            percentage = (missing_count / total_rows) * 100

            print(
                f"{column}: {missing_count} missing "
                f"({percentage:.2f}%)"
            )

In [ ]:
# Check duplicate identifier values in each typed DataFrame

primary_keys = {
    "stations": ("stations_df", stations_df, "station_id"),
    "observations": ("observations_df", observations_df, "observation_id"),
    "forecasts": ("forecasts_df", forecasts_df, "forecast_id"),
    "alerts": ("alerts_df", alerts_df, "alert_id")
}

for name, (df_name, df, key) in primary_keys.items():

    print(f"\n{'=' * 60}")
    print(f"DUPLICATE KEY CHECK: {name.upper()}")
    print(f"{'=' * 60}")

    duplicate_keys = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_groups = duplicate_keys.count()

    print(f"Duplicate key groups: {duplicate_groups}")

    if duplicate_groups > 0:
        print("\nDuplicate identifiers:")
        duplicate_keys.show(20, truncate=False)

In [ ]:
# Inspect all records associated with duplicate observation IDs

duplicate_observation_ids = (
    observations_df
    .groupBy("observation_id")
    .count()
    .filter(F.col("count") > 1)
    .select("observation_id")
)

duplicate_observations = (
    observations_df
    .join(duplicate_observation_ids, on="observation_id", how="inner")
    .orderBy("observation_id")
)

duplicate_observations.show(truncate=False)

In [ ]:
# Count exact duplicate rows across all columns

total_rows = observations_df.count()

distinct_rows = observations_df.distinct().count()

exact_duplicate_count = total_rows - distinct_rows

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicate rows:", exact_duplicate_count)

In [ ]:
# Inspect distinct categorical values and their frequencies

categorical_columns = {
    "stations": (stations_df, ["state"]),
    "observations": (observations_df, ["condition"]),
    "forecasts": (forecasts_df, ["forecast_condition"]),
    "alerts": (alerts_df, ["alert_type", "severity", "status"])
}

for dataset_name, (df, columns) in categorical_columns.items():

    print(f"\n{'=' * 60}")
    print(f"CATEGORICAL VALUE ASSESSMENT: {dataset_name.upper()}")
    print(f"{'=' * 60}")

    for column in columns:

        print(f"\nColumn: {column}")

        (
            df.groupBy(column)
            .count()
            .orderBy(F.desc("count"))
            .show(50, truncate=False)
        )

In [ ]:
# Preliminary numeric range assessment
#These are broad screening limits for this synthetic assignment, not official thresholds for every weather station.

numeric_ranges = {
    "stations": (
        stations_df,
        {"elevation_m": (-500, 9000)}
    ),
    "observations": (
        observations_df,
        {
            "temperature_c": (-50, 60),
            "humidity_pct": (0, 100),
            "rainfall_mm": (0, 1000),
            "wind_kph": (0, 400)
        }
    ),
    "forecasts": (
        forecasts_df,
        {
            "forecast_temp_c": (-50, 60),
            "forecast_rain_mm": (0, 1000)
        }
    )
}

for dataset_name, (df, columns) in numeric_ranges.items():

    print(f"\n{'=' * 60}")
    print(f"NUMERIC RANGE ASSESSMENT: {dataset_name.upper()}")
    print(f"{'=' * 60}")

    for column, (minimum, maximum) in columns.items():

        invalid_records = df.filter(
            F.col(column).isNotNull()
            & (
                (F.col(column) < minimum)
                | (F.col(column) > maximum)
            )
        )

        print(f"\nColumn: {column}")
        print(f"Allowed range: {minimum} to {maximum}")
        print(f"Out-of-range records: {invalid_records.count()}")

        invalid_records.select(
            "station_id", column
        ).show(10, truncate=False)

In [ ]:
# Inspect complete records containing invalid numeric measurements

invalid_observations = observations_df.filter(
    (F.col("temperature_c").isNotNull() &
     ~F.col("temperature_c").between(-50, 60))
    |
    (F.col("humidity_pct").isNotNull() &
     ~F.col("humidity_pct").between(0, 100))
    |
    (F.col("rainfall_mm").isNotNull() &
     ~F.col("rainfall_mm").between(0, 1000))
)

print("Records with at least one invalid numeric measurement:",
      invalid_observations.count())

invalid_observations.orderBy("observation_id").show(
    30, truncate=False
)

In [ ]:
# Inspect malformed timestamp values from raw DataFrames

timestamp_columns = {
    "observations": (raw_dfs["observations"], "observed_at"),
    "forecasts": (raw_dfs["forecasts"], "forecast_for"),
    "alerts": (raw_dfs["alerts"], "alert_time")
}

for dataset_name, (df, column) in timestamp_columns.items():

    invalid_condition = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) != "")
        & F.expr(f"try_cast(`{column}` AS TIMESTAMP)").isNull()
    )

    invalid_records = df.filter(invalid_condition)

    print(f"\n{dataset_name.upper()}")
    print("Malformed timestamp records:", invalid_records.count())

    invalid_records.show(truncate=False)

In [ ]:
# Check referential integrity against the station master

child_datasets = {
    "observations": observations_df,
    "forecasts": forecasts_df,
    "alerts": alerts_df
}

valid_station_ids = stations_df.select("station_id").distinct()

for dataset_name, df in child_datasets.items():

    orphan_records = df.join(
        valid_station_ids,
        on="station_id",
        how="left_anti"
    )

    print(f"\n{'=' * 60}")
    print(f"REFERENTIAL INTEGRITY: {dataset_name.upper()}")
    print(f"{'=' * 60}")

    print("Orphan records:", orphan_records.count())

    if orphan_records.count() > 0:
        orphan_records.show(20, truncate=False)

## **Standardization and Transformation**

In [ ]:

# Standardize categorical columns

observations_cleaned = observations_df.withColumn(
    "condition",
    F.lower(F.trim(F.col("condition")))
)

forecasts_cleaned = forecasts_df.withColumn(
    "forecast_condition",
    F.lower(F.trim(F.col("forecast_condition")))
)

alerts_cleaned = (
    alerts_df
    .withColumn("severity", F.lower(F.trim(F.col("severity"))))
    .withColumn("status", F.lower(F.trim(F.col("status"))))
)

In [ ]:
# Verify standardized categorical values

for dataset_name, df, columns in [
    ("OBSERVATIONS", observations_cleaned, ["condition"]),
    ("FORECASTS", forecasts_cleaned, ["forecast_condition"]),
    ("ALERTS", alerts_cleaned, ["severity", "status"])
]:

    print(f"\n{'=' * 50}")
    print(f"{dataset_name} — AFTER STANDARDIZATION")
    print(f"{'=' * 50}")

    for column in columns:
        print(f"\nColumn: {column}")

        df.groupBy(column).count().orderBy(
            F.desc("count")
        ).show(truncate=False)

## **Data Validation , Cleaning and Quarantinee**

In [ ]:

source_data = {
    "stations": stations_df,
    "observations": observations_cleaned,
    "forecasts": forecasts_cleaned,
    "alerts": alerts_cleaned
}

primary_keys = {
    "stations": "station_id",
    "observations": "observation_id",
    "forecasts": "forecast_id",
    "alerts": "alert_id"
}

duplicate_keys = {}

for name, key in primary_keys.items():
    duplicate_keys[name] = (
        source_data[name]
        .groupBy(key)
        .count()
        .filter(
            F.col(key).isNotNull()
            & (F.trim(F.col(key)) != "")
            & (F.col("count") > 1)
        )
        .select(key)
    )

for name, key in primary_keys.items():
    print(f"\n{name.upper()} — Duplicate {key} groups")
    duplicate_keys[name].show(truncate=False)


In [ ]:

# Inspect all records belonging to duplicate observation IDs

duplicate_observations = (
    observations_cleaned
    .join(
        duplicate_keys["observations"],
        on="observation_id",
        how="left_semi"
    )
    .orderBy("observation_id")
)

duplicate_observations.show(truncate=False)


In [ ]:

from pyspark.sql.window import Window

# Assign a row number within each group of completely identical records
duplicate_window = (
    Window
    .partitionBy(*observations_cleaned.columns)
    .orderBy(F.lit(1))
)

observations_ranked = observations_cleaned.withColumn(
    "_duplicate_rank",
    F.row_number().over(duplicate_window)
)

# Keep one copy of each record
observations_deduplicated = (
    observations_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank")
)

# Preserve only the additional copies for quarantine
exact_dupe_extras = (
    observations_ranked
    .filter(F.col("_duplicate_rank") > 1)
    .drop("_duplicate_rank")
    .withColumn("quarantine_reason", F.lit("duplicate_record"))
)

# Update the source DataFrame used in subsequent validation
source_data["observations"] = observations_deduplicated

print("Original observations:", observations_cleaned.count())
print("Retained observations:", observations_deduplicated.count())
print("Duplicate extras:", exact_dupe_extras.count())


In [ ]:

# Build valid station reference keys
station_keys = (
    stations_df
    .select(F.col("station_id").alias("_ref_id"))
    .filter(
        F.col("_ref_id").isNotNull()
        & (F.trim(F.col("_ref_id")) != "")
    )
    .distinct()
)


# Find non-empty references that do not exist in the station master
def unmatched_nonnullKeys(df, column, reference_keys):
    return (
        df.filter(
            F.col(column).isNotNull()
            & (F.trim(F.col(column)) != "")
        )
        .join(
            reference_keys,
            F.col(column) == F.col("_ref_id"),
            "left_anti"
        )
    )


# Check all datasets containing station_id
reference_issues = {}

for name in ["observations", "forecasts", "alerts"]:
    reference_issues[name] = unmatched_nonnullKeys(
        source_data[name],
        "station_id",
        station_keys
    )

    print(f"\n{name.upper()} — Unmatched station references")
    print("Count:", reference_issues[name].count())
    reference_issues[name].show(truncate=False)


In [ ]:

# Reference-integrity rules
reference_rules = {
    "observations": [
        ("station_id", station_keys)
    ],
    "forecasts": [
        ("station_id", station_keys)
    ],
    "alerts": [
        ("station_id", station_keys)
    ]
}

print("Reference rules configured.")


In [ ]:

# Define required-field and business-rule validation

invalid_rules = {
    "stations": lambda df: (
        F.col("station_id").isNull()
        | (F.trim(F.col("station_id")) == "")
        | F.col("station_name").isNull()
        | (F.trim(F.col("station_name")) == "")
        | F.col("state").isNull()
        | (F.trim(F.col("state")) == "")
        | (
            F.col("elevation_m").isNotNull()
            & (
                (F.col("elevation_m") < -500)
                | (F.col("elevation_m") > 9000)
            )
        )
    ),

    "observations": lambda df: (
        F.col("observation_id").isNull()
        | (F.trim(F.col("observation_id")) == "")
        | F.col("station_id").isNull()
        | (F.trim(F.col("station_id")) == "")
        | F.col("observed_at").isNull()
        | (
            F.col("temperature_c").isNotNull()
            & (
                (F.col("temperature_c") < -50)
                | (F.col("temperature_c") > 60)
            )
        )
        | (
            F.col("humidity_pct").isNotNull()
            & (
                (F.col("humidity_pct") < 0)
                | (F.col("humidity_pct") > 100)
            )
        )
        | (
            F.col("rainfall_mm").isNotNull()
            & (
                (F.col("rainfall_mm") < 0)
                | (F.col("rainfall_mm") > 1000)
            )
        )
        | (
            F.col("wind_kph").isNotNull()
            & (
                (F.col("wind_kph") < 0)
                | (F.col("wind_kph") > 400)
            )
        )
    ),

    "forecasts": lambda df: (
        F.col("forecast_id").isNull()
        | (F.trim(F.col("forecast_id")) == "")
        | F.col("station_id").isNull()
        | (F.trim(F.col("station_id")) == "")
        | F.col("forecast_for").isNull()
        | (
            F.col("forecast_temp_c").isNotNull()
            & (
                (F.col("forecast_temp_c") < -50)
                | (F.col("forecast_temp_c") > 60)
            )
        )
        | (
            F.col("forecast_rain_mm").isNotNull()
            & (
                (F.col("forecast_rain_mm") < 0)
                | (F.col("forecast_rain_mm") > 1000)
            )
        )
    ),

    "alerts": lambda df: (
        F.col("alert_id").isNull()
        | (F.trim(F.col("alert_id")) == "")
        | F.col("station_id").isNull()
        | (F.trim(F.col("station_id")) == "")
        | F.col("alert_time").isNull()
        | F.col("alert_type").isNull()
        | (F.trim(F.col("alert_type")) == "")
        | F.col("severity").isNull()
        | (F.trim(F.col("severity")) == "")
        | F.col("status").isNull()
        | (F.trim(F.col("status")) == "")
        | ~F.col("severity").isin("low", "medium", "high", "critical")
        | ~F.col("status").isin("issued", "expired", "resolved")
        | ~F.col("alert_type").isin(
            "flood_risk", "heavy_rain", "high_wind", "heat"
        )
    )
}

print("Invalid rules configured.")


In [ ]:

# Fields that should contain numeric values
numeric_fields = {
    "stations": ["elevation_m"],
    "observations": [
        "temperature_c", "humidity_pct", "rainfall_mm", "wind_kph"
    ],
    "forecasts": ["forecast_temp_c", "forecast_rain_mm"]
}

# Timestamp fields
timestamp_fields = {
    "observations": ["observed_at"],
    "forecasts": ["forecast_for"],
    "alerts": ["alert_time"]
}

# Check non-empty raw values that cannot be parsed as numbers
numeric_pattern = r"^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$"

malformed_counts = {}

for name, fields in numeric_fields.items():
    for field in fields:
        raw_col = F.trim(F.col(field))

        malformed_count = (
            raw_dfs[name]
            .filter(
                raw_col.isNotNull()
                & (raw_col != "")
                & (~raw_col.rlike(numeric_pattern))
            )
            .count()
        )

        malformed_counts[f"{name}.{field}"] = malformed_count

# Check non-empty raw timestamp values that cannot be parsed
for name, fields in timestamp_fields.items():
    for field in fields:
        raw_col = F.trim(F.col(field))

        malformed_count = (
            raw_dfs[name]
            .filter(
                raw_col.isNotNull()
                & (raw_col != "")
                & F.try_to_timestamp(raw_col).isNull()
            )
            .count()
        )

        malformed_counts[f"{name}.{field}"] = malformed_count

# Display field-level results
for field, count in malformed_counts.items():
    print(f"{field}: {count}")


In [ ]:

# Define fields to check in the original raw DataFrames

malformed_fields = {
    "observations": {
        "numeric": [
            "temperature_c", "humidity_pct", "rainfall_mm", "wind_kph"
        ],
        "timestamp": ["observed_at"]
    },
    "forecasts": {
        "numeric": ["forecast_temp_c", "forecast_rain_mm"],
        "timestamp": ["forecast_for"]
    },
    "alerts": {
        "numeric": [],
        "timestamp": ["alert_time"]
    }
}

malformed_records = {}

for name, field_groups in malformed_fields.items():
    raw_df = raw_dfs[name]
    key = primary_keys[name]

    reason_expressions = []

    # Check malformed numeric values
    for field in field_groups["numeric"]:
        raw_value = F.trim(F.col(field))

        invalid_value = (
            raw_value.isNotNull()
            & (raw_value != "")
            & (~raw_value.rlike(numeric_pattern))
        )

        reason_expressions.append(
            F.when(invalid_value, F.lit(f"malformed:{field}"))
        )

    # Check malformed timestamps
    for field in field_groups["timestamp"]:
        raw_value = F.trim(F.col(field))

        invalid_value = (
            raw_value.isNotNull()
            & (raw_value != "")
            & F.try_to_timestamp(raw_value).isNull()
        )

        reason_expressions.append(
            F.when(invalid_value, F.lit(f"malformed:{field}"))
        )

    malformed_records[name] = (
        raw_df
        .select(
            F.col(key),
            F.concat_ws("; ", *reason_expressions).alias(
                "quarantine_reason"
            )
        )
        .filter(F.col("quarantine_reason") != "")
    )

    print(f"\n{name.upper()} — Malformed records")
    print("Count:", malformed_records[name].count())
    malformed_records[name].show(truncate=False)


In [ ]:

# Collect records violating configured validation rules

quarantine_candidates = []

for name, df in source_data.items():

    key = primary_keys[name]
    invalid_records = (
        df
        .filter(invalid_rules[name](df))
        .select(
            F.col(key).cast("string").alias("record_id"),
            F.lit("validation_rule_violation").alias(
                "quarantine_reason"
            )
        )
        .withColumn("dataset", F.lit(name))
    )
    quarantine_candidates.append(invalid_records)


# Add records containing malformed non-empty values

for name, df in malformed_records.items():

    key = primary_keys[name]
    malformed_candidates = (
        df
        .select(
            F.col(key).cast("string").alias("record_id"),
            F.col("quarantine_reason")
        )
        .withColumn("dataset", F.lit(name))
    )
    quarantine_candidates.append(malformed_candidates)


# Add records with unmatched station references

for name, df in reference_issues.items():
    key = primary_keys[name]
    reference_candidates = (
        df
        .select(
            F.col(key).cast("string").alias("record_id"),
            F.lit("unmatched_station_id").alias(
                "quarantine_reason"
            )
        )
        .withColumn("dataset", F.lit(name))
    )
    quarantine_candidates.append(reference_candidates)


# Combine all validation failures

all_quarantine_candidates = quarantine_candidates[0]

for candidate_df in quarantine_candidates[1:]:
    all_quarantine_candidates = (
        all_quarantine_candidates.unionByName(candidate_df)
    )


print("Total quarantine candidate entries:",
      all_quarantine_candidates.count())
all_quarantine_candidates.show(20, truncate=False)


In [ ]:

# Consolidate multiple rejection reasons per record

quarantine_summary = (
    all_quarantine_candidates
    .groupBy("dataset", "record_id")
    .agg(
        F.concat_ws(
            "; ",
            F.sort_array(
                F.collect_set("quarantine_reason")
            )
        ).alias("quarantine_reason")
    )
)

print("Total uniquely identified quarantine records:",
      quarantine_summary.count())

quarantine_summary.orderBy(
    "dataset", "record_id"
).show(30, truncate=False)


In [ ]:
# Remove the existing reason column before adding the final reason
duplicate_quarantine = (
    exact_dupe_extras
    .drop("quarantine_reason")
    .withColumn("dataset", F.lit("observations"))
    .withColumnRenamed("observation_id", "record_id")
    .withColumn("quarantine_reason", F.lit("duplicate_record"))
)

print("Exact duplicate records to quarantine:",
      duplicate_quarantine.count())

duplicate_quarantine.show(truncate=False)

In [ ]:
clean_data = {}
quarantine_data = {}

for name, df in source_data.items():

    key = primary_keys[name]

    # Get the consolidated rejection reasons for this dataset
    reasons = (
        quarantine_summary
        .filter(F.col("dataset") == name)
        .select("record_id", "quarantine_reason")
    )

    # Match source records with their rejection reasons
    quarantine_data[name] = (
        df.alias("src")
        .join(
            reasons.alias("q"),
            F.col(f"src.{key}").cast("string") == F.col("q.record_id"),
            "inner"
        )
        .select(
            "src.*",
            F.col("q.quarantine_reason")
        )
    )

    # Retain records that have no rejection reason
    clean_data[name] = (
        df.alias("src")
        .join(
            reasons.alias("q"),
            F.col(f"src.{key}").cast("string") == F.col("q.record_id"),
            "left_anti"
        )
    )

In [ ]:
duplicate_observation_rows = (
    exact_dupe_extras
    .drop("quarantine_reason")
    .withColumn("quarantine_reason", F.lit("duplicate_record"))
)

quarantine_data["observations"] = (
    quarantine_data["observations"]
    .unionByName(duplicate_observation_rows)
)

In [ ]:
print("DATA QUALITY RECONCILIATION")

for name in source_data:

    original_count = dataframes[name].count()
    clean_count = clean_data[name].count()
    quarantine_count = quarantine_data[name].count()

    accounted_count = clean_count + quarantine_count

    print(f"\nDataset: {name}")
    print(f"Original records:       {original_count}")
    print(f"Clean records:          {clean_count}")
    print(f"Quarantined records:    {quarantine_count}")
    print(f"Accounted records:      {accounted_count}")
    print(f"Reconciliation:         {'PASS' if original_count == accounted_count else 'FAIL'}")

In [ ]:
for name in source_data:

    print(f"\n{'=' * 20} {name.upper()} {'=' * 20}")

    print("Clean records:")
    clean_data[name].show(5, truncate=False)

    print("Quarantined records:")
    quarantine_data[name].show(5, truncate=False)

In [ ]:
print("FINAL CLEAN DATA QUALITY CHECK")

for name, df in clean_data.items():

    key = primary_keys[name]

    print(f"\n{'=' * 15} {name.upper()} {'=' * 15}")

    # 1. Record count
    print("Record count:", df.count())

    # 2. Duplicate primary keys
    duplicate_key_count = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("Duplicate primary key groups:", duplicate_key_count)

    # 3. Null primary keys
    null_key_count = (
        df.filter(F.col(key).isNull())
        .count()
    )

    print("Null primary keys:", null_key_count)

    # 4. Null values across required fields
    required_fields = {
        "stations": ["station_id", "station_name", "state"],
        "observations": [
            "observation_id", "station_id", "observed_at"
        ],
        "forecasts": [
            "forecast_id", "station_id", "forecast_for"
        ],
        "alerts": [
            "alert_id", "station_id", "alert_time",
            "alert_type", "severity", "status"
        ]
    }

    for column in required_fields[name]:

        null_count = (
            df.filter(
                F.col(column).isNull() |
                (F.trim(F.col(column).cast("string")) == "")
            ).count()
        )

        print(f"Missing {column}: {null_count}")

In [ ]:
print("COMPREHENSIVE CLEAN DATA VALIDATION")

for name, df in clean_data.items():

    print(f"\n{'=' * 15} {name.upper()} {'=' * 15}")

    # 1. Configured business validation rules
    invalid_count = (
        df.filter(invalid_rules[name](df))
        .count()
    )

    print("Validation rule violations:", invalid_count)

    # 2. Malformed values
    if name in malformed_records:

        key = primary_keys[name]

        malformed_count = (
            df.select(key)
            .join(
                malformed_records[name].select(key),
                on=key,
                how="inner"
            )
            .count()
        )

    else:
        malformed_count = 0

    print("Malformed records:", malformed_count)

    # 3. Unmatched station references
    if name in reference_rules:

        unmatched_count = (
            unmatched_nonnullKeys(
                df,
                "station_id",
                station_keys
            ).count()
        )

        print("Unmatched station references:", unmatched_count)

## **Saving Clean and Quarantinee datasets**

In [ ]:
from pathlib import Path

output_base = Path("Data/processed")

for name in clean_data:

    clean_path = str(output_base / "clean" / name)
    quarantine_path = str(output_base / "quarantine" / name)

    # Write clean records
    (
        clean_data[name]
        .write
        .mode("overwrite")
        .parquet(clean_path)
    )

    # Write quarantined records
    (
        quarantine_data[name]
        .write
        .mode("overwrite")
        .parquet(quarantine_path)
    )

    print(f"{name.upper()} saved successfully")

In [ ]:
print("PARQUET STORAGE VALIDATION")

for name in clean_data:

    clean_path = str(output_base / "clean" / name)
    quarantine_path = str(output_base / "quarantine" / name)

    # Read stored Parquet datasets
    clean_readback = spark.read.parquet(clean_path)
    quarantine_readback = spark.read.parquet(quarantine_path)

    # Compare counts
    clean_original = clean_data[name].count()
    clean_stored = clean_readback.count()

    quarantine_original = quarantine_data[name].count()
    quarantine_stored = quarantine_readback.count()

    status = (
        clean_original == clean_stored
        and quarantine_original == quarantine_stored
    )

    print(f"\n{name.upper()}")
    print("Clean:", clean_original, "->", clean_stored)
    print("Quarantine:", quarantine_original, "->", quarantine_stored)
    print("Status:", "PASS" if status else "FAIL")